##**Quantitative Translational Imaging in Medicine Lab — Summer Scholar Program**

<div style="background-color:black; padding:20px; text-align:center; border-radius: 8px;">
    <img src="https://i0.wp.com/www.martinos.org/wp-content/uploads/2019/01/spark_no_fade.gif?fit=404%2C303&ssl=1" alt="Martinos Center Logo" width="400"/>
    <h1 style="color:white; font-family: sans-serif;">Martinos Center for Biomedical Imaging</h1>
</div>

# Day 2: MRI, CT, and Ultrasound

**Today's goal:** understand (conceptually) how MRI, CT, and ultrasound each work, and get
hands-on with cropping, filtering, thresholding, and measuring images from all three
modalities in code.

In [ ]:
# If you're running this in Google Colab, run this cell first to make sure
# scikit-image is installed. If you're running locally and already have it, this is
# harmless - it'll just say "already satisfied" and do nothing.
!pip install -q scikit-image

In [ ]:
# These functions generate our simulated MRI, CT, and ultrasound images.
# You don't need to understand every line - just run this cell once, then move on.
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from skimage.data import shepp_logan_phantom
from skimage.transform import resize

def make_synthetic_brain_mri(size=256, seed=1, add_tumor=True):
    rng = np.random.default_rng(seed)
    y, x = np.ogrid[:size, :size]
    cy, cx = size/2, size/2
    r = np.sqrt((y-cy)**2 + (x-cx)**2)
    img = np.zeros((size, size))
    R = size*0.42
    img[(r < R) & (r > R*0.92)] = 0.95
    tissue_mask = r < R*0.90
    base = 0.55 + 0.05*gaussian_filter(rng.standard_normal((size,size)), 8)
    img[tissue_mask] = base[tissue_mask]
    white_mask = r < R*0.60
    img[white_mask] = 0.75 + 0.03*gaussian_filter(rng.standard_normal((size,size)), 8)[white_mask]
    for sign in [-1, 1]:
        ey, ex = cy, cx + sign*size*0.09
        ell = ((y-ey)/(size*0.10))**2 + ((x-ex)/(size*0.05))**2
        img[ell < 1] = 0.15
    if add_tumor:
        ty, tx = cy - size*0.15, cx + size*0.20
        ell = ((y-ty)/(size*0.045))**2 + ((x-tx)/(size*0.055))**2
        img[ell < 1] = 0.92
    img = gaussian_filter(img, 1.0)
    img += rng.normal(0, 0.02, img.shape)
    img[r >= R] = np.clip(img[r >= R], 0, 0.03)
    return np.clip(img, 0, 1)

def make_ct_phantom(size=256):
    p = shepp_logan_phantom()
    p = resize(p, (size, size), anti_aliasing=True)
    hu = -1000 + p * 2000
    return np.clip(hu, -1000, 1500)

def make_ultrasound(size=256, seed=2):
    rng = np.random.default_rng(seed)
    y, x = np.ogrid[:size, :size]
    depth_gain = np.clip(1.2 - (y/size)*0.8, 0.2, 1.2)
    base = 0.4 * depth_gain * np.ones((size, size))
    speckle = rng.gamma(shape=4, scale=0.25, size=(size, size))
    img = base * speckle
    cy, cx, cr = size*0.55, size*0.5, size*0.08
    r = np.sqrt((y-cy)**2 + (x-cx)**2)
    img[r < cr] = rng.gamma(4, 0.03, img.shape)[r < cr]
    enhance_mask = (x > cx-cr) & (x < cx+cr) & (y > cy+cr) & (y < cy+cr*3)
    img[enhance_mask] *= 1.4
    img = gaussian_filter(img, 0.7)
    return np.clip(img, 0, 1.5)

print("Ready to go!")

---
## Part 1: MRI

**Before you start, watch this short video:**
📺 [How does an MRI work? — animated explainer, ~5 min](https://www.youtube.com/watch?v=nFkBhUYynUw&t=149s)

Optional, if you want more depth on the physics afterward:
📖 [Khan Academy: Magnetic Resonance Imaging (MRI) — short article](https://www.khanacademy.org/test-prep/mcat/physical-processes/proton-nuclear-magnetic-resonance/a/magnetic-resonance-imaging-mri)

**Quick recap of the physics**:
- Your body is full of hydrogen atoms (mostly from water and fat). Each one acts like a tiny
  spinning magnet.
- The MRI machine's big magnet lines all these tiny magnets up.
- A radio pulse knocks them out of alignment. As they "relax" back into alignment, they give
  off a radio signal the machine detects.
- **How fast** they relax depends on the tissue they're in — fat relaxes differently than
  water, which relaxes differently than a tumor. This is where image *contrast* comes from.
- **T1** and **T2** are two ways of measuring/weighting this relaxation, and they make
  different tissues look bright or dark.

**Journal prompt before moving on:** in 2-3 sentences, explain to a friend what "contrast" in
an MRI image actually comes from. This is a good check that the video's concept landed.

In [ ]:
# Journal entry:
#

### 1.1 T1-weighted vs T2-weighted — same anatomy, different contrast

In real MRI, the same patient's head can be scanned with different settings ("sequences")
that emphasize different tissue properties. Fluid (like the fluid-filled ventricles in the
brain) is a great example: it's **dark** on T1-weighted images and **bright** on
T2-weighted images.

We'll approximate this with a simplified simulation: we'll take our T1-like image and
**invert** it (flip bright and dark) to get a rough T2-like version. This isn't physically
exact — real contrast comes from actual relaxation-time differences, not a simple flip — but
it captures the key visual idea.

In [ ]:
t1_like = make_synthetic_brain_mri(seed=1)

plt.imshow(t1_like, cmap='gray')
plt.title('"T1-weighted" (fluid = dark)')
plt.axis('off')
plt.show()

In [ ]:
# 1.0 - t1_like flips every pixel value: a pixel that was 0.9 becomes 0.1, and so on
t2_like = 1.0 - t1_like

# the skull bone should stay dark in both versions (bone stays dark on T1 and T2),
# so we fix that up here rather than leaving it inverted
skull_mask = t1_like > 0.85   # find the bright skull pixels in the original
t2_like[skull_mask] = 0.1     # force them dark in our T2 version

plt.imshow(t2_like, cmap='gray')
plt.title('"T2-weighted" (fluid = bright)')
plt.axis('off')
plt.show()

Look at the the two dark shapes near the center in the T1 image.
What happened to them in the T2 image?

This matters clinically: radiologists choose T1 vs T2 (or other sequences) depending on what
they're looking for. Fluid, swelling, and many tumors show up much more clearly on T2.

### 1.2 Cropping — zooming in on a Region of Interest (ROI)

Researchers rarely want to analyze a whole image — usually just a specific region, called a
**Region of Interest (ROI)**. Cropping an image is just slicing the number-grid (remember
slicing from yesterday: `image[row_start:row_end, col_start:col_end]`).

In [ ]:
print("Full image shape:", t1_like.shape)

# Crop out rows 60-180 and columns 60-180
roi = t1_like[60:180, 60:180]

print("Cropped ROI shape:", roi.shape)

In [ ]:
# Let's see the full image with a red box showing exactly where we cropped
plt.imshow(t1_like, cmap='gray')
plt.plot([60,180,180,60,60], [60,60,180,180,60], 'r-', linewidth=2)
plt.title('Full image (red box = crop area)')
plt.axis('off')
plt.show()

In [ ]:
# And here's just the cropped piece
plt.imshow(roi, cmap='gray')
plt.title('Cropped ROI')
plt.axis('off')
plt.show()

**Try it:** change the four numbers (60, 180, 60, 180) in the cropping cell above to
zoom into the bright "lesion" spot near the top-right of the brain. You'll need to estimate
its row/column location by looking at the full image first — that's a normal part of working
with images like this.

### 1.3 Filtering — reducing noise

Real MRI images always have some random noise. A common first step in analysis is
**smoothing** (blurring) the image slightly, using a technique called **Gaussian filtering**.
Let's see the effect, one blur amount at a time.

In [ ]:
smoothed_a_little = gaussian_filter(t1_like, sigma=1)
smoothed_a_lot = gaussian_filter(t1_like, sigma=6)

fig, axes = plt.subplots(1, 3, figsize=(12,4))
axes[0].imshow(t1_like, cmap='gray'); axes[0].set_title('Original (sigma=0)')
axes[1].imshow(smoothed_a_little, cmap='gray'); axes[1].set_title('sigma=1')
axes[2].imshow(smoothed_a_lot, cmap='gray'); axes[2].set_title('sigma=6')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

**Question:** as blur increases, the image gets smoother, but also loses detail. At
what blur amount does the "lesion" spot become hard to distinguish from normal tissue? Try
changing `sigma=6` to other numbers (try 2, 3, 10) and re-run to explore.

This trade-off (noise reduction vs. detail loss) comes up constantly in image analysis
research.

### 1.4 Measuring signal intensity — the beginning of quantitative imaging

Our lab is a **quantitative** imaging lab — that means we
*measure* things from medical images. The simplest measurement: average pixel intensity within a
region, using `.mean()`.

In [ ]:
# We picked out three small regions ahead of time by looking at the image.
white_matter_patch = t1_like[118:138, 118:138]
ventricle_patch = t1_like[118:138, 95:112]
lesion_patch = t1_like[70:90, 155:175]

print("Mean intensity, white matter:", round(white_matter_patch.mean(), 3))
print("Mean intensity, ventricle:   ", round(ventricle_patch.mean(), 3))
print("Mean intensity, lesion:      ", round(lesion_patch.mean(), 3))

In [ ]:
# Let's visualize exactly where these three regions are
fig, ax = plt.subplots(figsize=(6,6))
ax.imshow(t1_like, cmap='gray')

ax.plot([118,138,138,118,118], [118,118,138,138,118], color='yellow', linewidth=2, label='white matter')
ax.plot([95,112,112,95,95], [118,118,138,138,118], color='cyan', linewidth=2, label='ventricle')
ax.plot([155,175,175,155,155], [70,70,90,90,70], color='red', linewidth=2, label='lesion')

ax.legend(loc='upper left', fontsize=8)
ax.axis('off')
plt.show()

The "lesion" region has a much higher mean intensity than
normal tissue. In real research, comparing signal intensity between a region of interest and
normal tissue is one of the most basic but genuinely useful tools for studying disease on
MRI.

### 1.5 Histograms — seeing the distribution of all pixel values at once

Instead of looking at just a few points, we can plot a **histogram**: how many pixels have
each intensity value. This is a core tool in quantitative imaging.

In [ ]:
plt.figure(figsize=(7,4))
plt.hist(t1_like.ravel(), bins=50, color='steelblue')
plt.xlabel('Pixel intensity')
plt.ylabel('Number of pixels')
plt.title('Histogram of MRI intensities')
plt.show()

**Question:** why do you think there's such a tall spike near 0? (Hint: think about how
much of the image is background vs. brain tissue.)

---
## Part 2: CT

**Before you start, watch this short video:**
📺 [How Does a CT Scan Work? — NIH/NIBIB, 60 Seconds of Science](https://www.youtube.com/watch?v=l9swbAtRRbg)

A CT scanner rotates an X-ray source around the patient, taking many 2D projections from
different angles. A computer algorithm combines all those projections into a
cross-sectional image. This is very different from a regular X-ray, which is just one flat
projection.

Denser material (like bone) absorbs more X-rays, so it shows up bright. Air absorbs almost
none, so it's dark.

In [ ]:
ct = make_ct_phantom()

plt.figure(figsize=(6,6))
plt.imshow(ct, cmap='gray')
plt.colorbar(label='Hounsfield Units (HU)')
plt.title('CT phantom')
plt.show()

### 2.1 Segmentation — separating tissue types by their HU value

Because CT values are physically calibrated (Hounsfield Units), we can write simple code
that automatically identifies certain tissue types just by their number range. This is called
**thresholding**, and it's the simplest form of image **segmentation**.

The `>` symbol compares every pixel to a number and gives back `True` or `False` for each
one — this creates a **mask**: a grid of True/False values the same size as the image.

In [ ]:
# Compare every pixel in ct to 300. Pixels above 300 HU become True (bone-like).
bone_mask = ct > 300

plt.figure(figsize=(6,6))
plt.imshow(bone_mask, cmap='gray')
plt.title('Pixels above 300 HU (bone-like)')
plt.show()

In [ ]:
# .sum() on a True/False grid counts how many Trues there are
print("Number of 'bone' pixels:", bone_mask.sum())
print("Total pixels in image:", bone_mask.size)
print("Percent of image:", round(100 * bone_mask.sum() / bone_mask.size, 1), "%")

**Try it:** change the threshold value (currently `300`) to something lower, like `100`,
or higher, like `800`, and re-run both cells above. What happens at very low thresholds?
Very high ones?

This is genuinely how a lot of real segmentation work starts, even if modern tools (like the
AI you'll build later this week) do something far more sophisticated on top of this basic
idea.

### 2.2 Overlaying a segmentation on the original image

A segmentation mask by itself isn't very useful visually — usually we overlay it in color on
top of the original grayscale image so a human can quickly check whether it looks right.

In [ ]:
fig, ax = plt.subplots(figsize=(6,6))
ax.imshow(ct, cmap='gray')

# build a red, semi-transparent overlay that only shows up where bone_mask is True
overlay = np.zeros((*bone_mask.shape, 4))  # 4 = red, green, blue, transparency
overlay[bone_mask] = [1, 0, 0, 0.5]         # red, 50% transparent, only at True locations

ax.imshow(overlay)
ax.set_title('Bone segmentation overlaid on CT')
ax.axis('off')
plt.show()

---
## Part 3: Ultrasound

**Before you start, watch this short video:**
📺 [How to See With Sound — TED-Ed, ~5 min](https://www.youtube.com/watch?v=4JLNb8-LOB0)

Ultrasound sends high-frequency sound waves into the body and listens for echoes. Different
tissue boundaries reflect sound differently, and the machine uses the *timing* and
*strength* of echoes to build an image. Because it uses sound instead of ionizing radiation,
it's considered very safe — that's why it's used so often in pregnancy.

Ultrasound images look very different from MRI/CT: grainy, with a characteristic
"speckle" texture.

In [ ]:
us = make_ultrasound()

plt.figure(figsize=(6,6))
plt.imshow(us, cmap='gray')
plt.title('Simulated ultrasound')
plt.show()

### 3.1 Artifacts

Every imaging modality has characteristic **artifacts**: patterns in the image that come from
the physics of the machine, not the patient's real anatomy. Learning to recognize them is a
core skill for anyone working with medical images.

Look closely at the dark circular region (a simulated fluid-filled cyst) in the ultrasound
image above. Notice the **brighter column right underneath it**? That's called **posterior
acoustic enhancement** — a real ultrasound artifact that happens because sound travels
through fluid very easily, so more sound energy reaches the tissue behind it, making that
region look artificially brighter.

In [ ]:
# Let's measure it: compare average brightness in a region below the cyst
# vs. a region at the same depth but off to the side (no cyst above it)

below_cyst = us[160:220, 118:138]
control_region = us[160:220, 40:60]

print("Mean brightness below cyst:    ", round(below_cyst.mean(), 3))
print("Mean brightness (control area):", round(control_region.mean(), 3))
print("Ratio:", round(below_cyst.mean() / control_region.mean(), 2), "x brighter")

If a researcher (or an automated algorithm) didn't know
about this artifact, what mistake might they make when analyzing this image?

This is a great example of why understanding the *physics* behind an imaging modality
matters, even when you're "just" writing analysis code — the numbers only mean what you think
they mean if you understand where they came from.

---
## Part 4: Comparing all three modalities

Let's measure noise level in a roughly uniform region for each modality we've seen today,
using standard deviation (`.std()`) as a simple noise metric — higher std in a flat region
means more noise.

In [ ]:
mri_patch = t1_like[118:138, 118:138]
ct_patch = ct[170:190, 170:190]
us_patch = us[60:80, 40:60]

print("MRI        std (noise):", round(mri_patch.std(), 4))
print("CT         std (noise):", round(ct_patch.std(), 4))
print("Ultrasound std (noise):", round(us_patch.std(), 4))

**Question:** which modality has the highest relative noise? Does that match what your
eyes told you when we looked at all three side-by-side yesterday on Day 1?

**Journal prompt:** artifacts are basically "the machine tricking you a little." Can you
think of an everyday, non-medical example of something similar — a situation where a
measurement or a photo can mislead you if you don't know how it was taken?

In [ ]:
# Journal entry:
#

## Looking ahead: from hand-written rules to AI

Today you've been writing your own rules to find things in images — a threshold for
bone, a threshold for a lesion. You had to *already know* roughly what number range to look
for, and you had to hand-tune it.

That works fine for simple cases. But real diagnostic problems are often far too subtle for a
human to write a simple threshold rule — the pattern that separates "healthy" from "diseased"
tissue can be spread across thousands of pixels in ways no human would ever spot by eye.

**Later this week**, instead of writing the rule yourself, you'll show a computer thousands
of labeled examples and let it **learn the rule on its own** — this is the core idea behind
AI/deep learning in medical imaging, and an active area of real research. You'll build, train, and evaluate a real image-classification AI model
on Thursday and Friday.